# Planning Intervention Experiment — Paper 1 Causality Section

**Goal**: confirm forward planning is CAUSAL (not just correlational) by patching the planned answer feature and measuring if the final answer flips.

**Protocol** (Anthropic-style):
1. For N test rollouts, identify `source_letter` (what model actually answered) and pick `target_letter` (different letter).
2. From our probe weights, get direction vectors `d_source` and `d_target` in L11 early-pooled space.
3. Re-generate response with hook: at token 10 of response, **subtract** `d_source` and **add** `d_target` to L11 residual.
4. Let model continue generating freely.
5. Extract new `\boxed{letter}` from output.
6. Measure: % of rollouts where final letter matches `target_letter` (success) vs stayed as `source_letter` (failed) vs other letter (partial shift).

**Expected** (if planning is causal):
- ≥ 30% of patches flip to target (Anthropic saw 70% on poetry — reasoning may be harder)
- ≤ 50% stay on source (indicating planning is LOAD-BEARING)
- Control: patching with random direction should NOT flip answer

**Compute**: ~60 rollouts × 3 target letters × ~90s/gen = **~4.5h** on RTX 6000 Blackwell 96GB or B200. ~$15-25.

**Pre-reqs**: this notebook needs the Stage B corpus + rollout tokens (same setup as detection notebook). Can reuse cached artifacts from `/content/planning/`.

## Cell 1 — Setup + load model

In [ ]:
import sys, subprocess, os, shutil, site
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)

    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)

    # fla and causal-conv1d are "nice to have" for speed — never block on them
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)

    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, pickle, time, random
import numpy as np
import torch
import torch.nn.functional as F
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/intervention')
OUT.mkdir(exist_ok=True)
print('setup done')


## Cell 2 — Load model + probe direction

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
PATCH_LAYER = 11  # use L11 — best T=10 accuracy
PATCH_POSITION = 10  # patch the 10th response token

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Load or regen probe — reuse cached if present
PROBE_CACHE = OUT / 'probe_directions.pkl'
if PROBE_CACHE.exists():
    with open(PROBE_CACHE, 'rb') as f:
        probe_data = pickle.load(f)
    scaler, pca, logreg, directions_by_letter, labels_letter, rollout_info, response_tokens_all = (
        probe_data[k] for k in ['scaler','pca','logreg','directions','labels','rollouts','tokens'])
    print(f'\u2705 loaded cached probe + {len(labels_letter)} rollouts')
else:
    # Download Stage B + fit
    corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                                repo_type='dataset', cache_dir='/content/cache')
    shard_dir = Path(corpus) / 'shards'
    shards = sorted(shard_dir.glob('q*.safetensors'))
    print(f'{len(shards)} shards')
    
    MIN_LEN = 200
    PATCH_WINDOW = 10
    pooled = []; labels_letter = []; rollout_info = []; response_tokens_all = []
    for shard in tqdm(shards, desc='probe-fit'):
        with safe_open(str(shard), framework='pt') as f:
            meta = f.metadata()
            offs = json.loads(meta['offsets'])[f'L{PATCH_LAYER}']
            rollouts = json.loads(meta['rollouts'])
            rts = json.loads(meta['response_tokens'])
            q_options = json.loads(meta['options'])
            q_idx = int(meta['question_global_idx'])
            acts = f.get_tensor(f'acts_L{PATCH_LAYER}')
            for r_idx, r in enumerate(rollouts):
                if r['pred'] is None or r['response_len'] < MIN_LEN:
                    continue
                rollout_acts = acts[offs[r_idx]:offs[r_idx+1]].float().numpy()
                pooled.append(rollout_acts[:PATCH_WINDOW].mean(axis=0))
                labels_letter.append(r['pred'])
                rollout_info.append(dict(
                    question=meta['question'], options=q_options, gold=meta['gold_letter'],
                    pred=r['pred'], correct=r['correct'], question_idx=q_idx, rollout_id=r['rollout_id']))
                response_tokens_all.append(rts[r_idx])
    
    pooled = np.stack(pooled)
    labels_letter = np.array(labels_letter)
    print(f'Extracted {len(pooled)} rollouts; letter dist: {dict((l, int((labels_letter==l).sum())) for l in sorted(set(labels_letter)))}')
    
    # Fit probe on ALL data (no test split — we'll intervene on held-out rollouts)
    letter_to_int = {l: i for i, l in enumerate(sorted(set(labels_letter)))}
    y = np.array([letter_to_int[l] for l in labels_letter])
    scaler = StandardScaler().fit(pooled)
    pca = PCA(n_components=128, random_state=42).fit(scaler.transform(pooled))
    logreg = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(pca.transform(scaler.transform(pooled)), y)
    
    # Extract direction per letter: weight vector back-projected to activation space
    # w_raw = pca.components_.T @ logreg.coef_[letter_idx]  — this is in scaled space
    # Convert to unscaled: direction = (pca.components_.T @ coef) / scaler.scale_
    directions_by_letter = {}
    for letter, int_idx in letter_to_int.items():
        coef_pca = logreg.coef_[int_idx]  # [128]
        # back-project through PCA: direction in scaled space
        direction_scaled = pca.components_.T @ coef_pca  # [d_model]
        # Don't un-scale — we want direction in scaled space for easier manipulation
        # But to patch activations directly, we need in original space.
        # Since patch = current + alpha * (target - source), the scaling cancels if we
        # apply both as unscaled directions. Let's use unscaled: direction_raw = direction_scaled * scaler.scale_
        direction_raw = direction_scaled * scaler.scale_
        # Normalize
        direction_raw = direction_raw / (np.linalg.norm(direction_raw) + 1e-8)
        directions_by_letter[letter] = direction_raw.astype(np.float32)
    
    with open(PROBE_CACHE, 'wb') as f:
        pickle.dump(dict(scaler=scaler, pca=pca, logreg=logreg,
                         directions=directions_by_letter, labels=labels_letter,
                         rollouts=rollout_info, tokens=response_tokens_all), f)
    print(f'\u2705 cached probe + directions')

print(f'Letters: {list(directions_by_letter.keys())}')

## Cell 3 — Hook utilities for patching

We'll use `forward_hook` on layer L_PATCH to modify the residual at a specific token position on demand.

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

class PatchController:
    """State machine to patch residual at a specific token position during generation.
    On each forward call: if we're at the patch step, add the patch vector to the correct token.
    """
    def __init__(self):
        self.patch_vector = None  # [d_model] to ADD at target position
        self.prompt_len = None
        self.target_response_pos = None  # which RESPONSE token to patch
        self.active = False
        self.applied = False
    
    def start(self, patch_vec, prompt_len, target_response_pos):
        self.patch_vector = torch.from_numpy(patch_vec).to('cuda', torch.bfloat16)
        self.prompt_len = prompt_len
        self.target_response_pos = target_response_pos
        self.active = True
        self.applied = False
    
    def stop(self):
        self.active = False
        self.patch_vector = None
    
    def make_hook(self):
        def hook(module, inp, out):
            if not self.active or self.applied:
                return out
            hidden = out[0] if isinstance(out, tuple) else out
            # hidden shape: [B, T_seen, d]
            # During incremental generation with KV cache, T_seen may be 1 (just new token)
            # During initial prompt processing, T_seen = full prompt length
            # We want to patch at absolute position = prompt_len + target_response_pos
            target_abs_pos = self.prompt_len + self.target_response_pos
            if hidden.shape[1] > target_abs_pos:
                # Initial pass through full sequence (e.g., re-run on full gen)
                hidden = hidden.clone()
                hidden[:, target_abs_pos, :] = hidden[:, target_abs_pos, :] + self.patch_vector
                self.applied = True
                if isinstance(out, tuple):
                    return (hidden,) + out[1:]
                return hidden
            # Otherwise: generation step — check if THIS is the target absolute pos
            # We track via prompt_len + target_response_pos. Since we don't easily know
            # the current absolute pos during generation, fall back to: patch at the first
            # call where shape[1] == target_abs_pos+1 (i.e. this is the generation step for target_response_pos)
            return out
        return hook

controller = PatchController()
layer_module = get_layer_module(model, PATCH_LAYER)
hook_handle = layer_module.register_forward_hook(controller.make_hook())
print(f'\u2705 hook installed on L{PATCH_LAYER}')

## Cell 4 — Intervention function: re-generate with patch

Strategy: use forced generation. Given original first 10 tokens from rollout, we:
1. Build prompt = question prompt + first 10 response tokens (teacher-forced)
2. Activate patch controller: at position `prompt_len + 10`, ADD patch vector
3. Generate remaining tokens freely
4. Extract final \boxed{letter}

In [ ]:
BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')

def extract_letter(c, n_opts):
    m = list(BOXED_RE.finditer(c))
    if m:
        l = m[-1].group(1).upper()
        if ord(l)-ord('A') < n_opts: return l
    tail = c[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands and ord(cands[-1])-ord('A') < n_opts: return cands[-1]
    return None

def format_prompt(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = (
        "Answer the following multiple-choice question. Reason step by step, "
        "then put the letter of the correct answer in \\boxed{}.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

def generate_with_patch(prompt, forced_response_prefix_tokens, patch_vec, max_new=1024):
    """Given prompt + first N response tokens, generate rest. Apply patch at position N (within response).
    If patch_vec is None, generate without patching (control).
    """
    enc = tok(prompt, return_tensors='pt').to('cuda')
    prompt_len = enc.input_ids.shape[1]
    # Build full input: prompt + forced_prefix
    forced_ids = torch.tensor([forced_response_prefix_tokens], device='cuda')
    full_input = torch.cat([enc.input_ids, forced_ids], dim=1)
    full_attn = torch.ones_like(full_input)
    # Position for patch: prompt_len + (len(forced)-1) — the LAST forced token's L11 acts
    patch_pos_within_response = len(forced_response_prefix_tokens) - 1  # 0-indexed
    
    if patch_vec is not None:
        controller.start(patch_vec, prompt_len, patch_pos_within_response)
    else:
        controller.stop()
    try:
        with torch.no_grad():
            out = model.generate(
                input_ids=full_input, attention_mask=full_attn,
                max_new_tokens=max_new, do_sample=False,
                pad_token_id=tok.pad_token_id, use_cache=True,
            )
    finally:
        controller.stop()
    # Extract generated part = full_input + generated (starts from full_input end)
    generated = out[0, full_input.shape[1]:]
    full_response = torch.cat([forced_ids[0], generated])
    text = tok.decode(full_response, skip_special_tokens=True)
    return text

print('\u2705 intervention function ready')

## Cell 5 — Run intervention on sample

Protocol:
- Sample N=30 CORRECT rollouts (we want to check: can we flip a confidently-correct model?)
- For each: get `source_letter` (= pred). Pick `target_letter` = random different letter.
- patch_vec = alpha * (d_target - d_source), alpha determines strength (start 3.0)
- Check: does the re-generated final letter == target? source? other?

Control:
- Also run WITHOUT patch (patch_vec=None) → should recover original answer (sanity)
- Also run with RANDOM direction (same norm as patch) → should NOT systematically flip (null)

In [ ]:
# Select N CORRECT rollouts with high probe confidence (model's plan was clear)
correct_mask = np.array([r['correct'] for r in rollout_info], dtype=bool)
confident_idx = np.where(correct_mask)[0]  # all correct rollouts
np.random.seed(42)
np.random.shuffle(confident_idx)

N_INTERVENTIONS = 20  # start small
PATCH_ALPHA = 5.0  # strength of direction shift — Anthropic used ~2-5x norm

results = []
t0 = time.time()
for i, rollout_idx in enumerate(tqdm(confident_idx[:N_INTERVENTIONS], desc='intervene')):
    info = rollout_info[rollout_idx]
    source_letter = info['pred']
    letters_avail = [l for l in directions_by_letter if l != source_letter and ord(l)-ord('A') < len(info['options'])]
    if not letters_avail: continue
    target_letter = random.choice(letters_avail)
    
    prompt = format_prompt(info['question'], info['options'])
    forced_prefix = response_tokens_all[rollout_idx][:PATCH_POSITION]  # first 10 response tokens
    
    # Compute patch vector
    d_source = directions_by_letter[source_letter]
    d_target = directions_by_letter[target_letter]
    patch_vec = PATCH_ALPHA * (d_target - d_source)
    
    # Baseline: no patch (should produce same letter as original pred)
    text_baseline = generate_with_patch(prompt, forced_prefix, None, max_new=1024)
    letter_baseline = extract_letter(text_baseline, len(info['options']))
    
    # Patched run
    text_patched = generate_with_patch(prompt, forced_prefix, patch_vec, max_new=1024)
    letter_patched = extract_letter(text_patched, len(info['options']))
    
    # Null control: random direction same norm
    random_dir = np.random.randn(len(d_source)).astype(np.float32)
    random_dir = random_dir / np.linalg.norm(random_dir) * PATCH_ALPHA
    text_random = generate_with_patch(prompt, forced_prefix, random_dir, max_new=1024)
    letter_random = extract_letter(text_random, len(info['options']))
    
    results.append(dict(
        idx=int(rollout_idx),
        source=source_letter, target=target_letter,
        baseline=letter_baseline, patched=letter_patched, random=letter_random,
        gold=info['gold'],
    ))
    if (i+1) % 5 == 0:
        n_flip = sum(1 for r in results if r['patched'] == r['target'])
        n_keep = sum(1 for r in results if r['patched'] == r['source'])
        print(f'  [{i+1}/{N_INTERVENTIONS}] flipped to target: {n_flip}/{len(results)} | kept source: {n_keep}')

with open(OUT/'intervention_results.json', 'w') as f:
    json.dump(dict(alpha=PATCH_ALPHA, n=len(results), results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n\u2705 done in {(time.time()-t0)/60:.1f} min')

## Cell 6 — Analysis

In [ ]:
n = len(results)

# Categorize
def categorize(r, which):
    l = r[which]
    if l == r['source']: return 'source'
    if l == r['target']: return 'target'
    if l is None: return 'invalid'
    return 'other'

for which in ['baseline', 'patched', 'random']:
    c = {'source':0,'target':0,'other':0,'invalid':0}
    for r in results:
        c[categorize(r, which)] += 1
    print(f'\n{which.upper()} outcomes:')
    for cat, cnt in c.items():
        print(f'  {cat:<10} {cnt:>3}/{n}  ({100*cnt/n:.1f}%)')

# Main claim
flip_rate = sum(1 for r in results if r['patched'] == r['target']) / n
keep_rate = sum(1 for r in results if r['patched'] == r['source']) / n
null_flip = sum(1 for r in results if r['random'] == r['target']) / n

print(f'\n{"="*50}')
print(f'CAUSALITY VERDICT')
print(f'{"="*50}')
print(f'Directional patch flip rate:  {100*flip_rate:.1f}%')
print(f'Random patch flip rate:       {100*null_flip:.1f}%')
print(f'Effect (direct - null):       {100*(flip_rate-null_flip):+.1f}pp')
print(f'Source kept (patch failed):   {100*keep_rate:.1f}%')
print()
if flip_rate - null_flip > 0.30:
    print('\u2b50 STRONG CAUSAL EVIDENCE — planning feature is intervenable')
elif flip_rate - null_flip > 0.15:
    print('\u2705 Moderate causal evidence')
elif flip_rate - null_flip > 0.05:
    print('\u26a0\ufe0f Weak causal signal — may need stronger alpha or different layer')
else:
    print('\u274c No causal effect — activation signal is epiphenomenal')

# Save final report
with open(OUT/'intervention_report.json', 'w') as f:
    json.dump(dict(
        alpha=PATCH_ALPHA, n=n,
        flip_rate=float(flip_rate),
        null_flip_rate=float(null_flip),
        causal_effect_pp=float(100*(flip_rate-null_flip)),
        keep_rate=float(keep_rate),
        results=results,
    ), f, indent=2)
print(f'\n\u2705 saved {OUT}/intervention_report.json')

## Cell 7 — V2 stronger patch (run only if v1 flips == 0)

Changes:
- **L_PATCH = 17** (best detection AUROC in controls)
- **PATCH_ALPHA = 12.0** (2.4x stronger)
- **PATCH_POSITION = 15** (deeper into latent plan)

Re-installs hook at new layer, same function, same 20 rollouts for paired comparison.


In [ ]:
# V2 CONFIG — stronger patch, different layer, deeper position
L_PATCH_V2 = 17
PATCH_ALPHA_V2 = 12.0
PATCH_POSITION_V2 = 15

# Uninstall old hook, install new one at L17
controller.stop()
if hasattr(controller, 'handle') and controller.handle is not None:
    controller.handle.remove()
    controller.handle = None

layer_v2 = get_layer_module(model, L_PATCH_V2)
controller.handle = layer_v2.register_forward_hook(controller._hook)
print(f'hook moved to L{L_PATCH_V2}')

# Rebuild probe directions at L17
# Re-fit probe using L17 activations from Stage B cache (stored during Cell 4 probe fit)
# Assumes activations_by_layer[17] was cached — if not, needs Stage B reload
assert 17 in probe_by_layer, 'L17 probe not fit in Cell 4 — re-run probe fit with L=[11,17]'

pca_v2 = pca_by_layer[17]
probe_v2 = probe_by_layer[17]
W_v2 = probe_v2.coef_  # (n_classes, n_pca)
# Back-project to model space
directions_by_letter_v2 = {}
for i, letter in enumerate(letters):
    d_pca = W_v2[i]
    d_model_space = pca_v2.components_.T @ d_pca  # (d_model,)
    d_model_space = d_model_space / (np.linalg.norm(d_model_space) + 1e-8)
    directions_by_letter_v2[letter] = d_model_space.astype(np.float32)
print(f'✅ V2 directions ready: L{L_PATCH_V2}, alpha={PATCH_ALPHA_V2}, pos={PATCH_POSITION_V2}')


In [ ]:
# Run V2 on SAME 20 rollouts for paired comparison
results_v2 = []
t0 = time.time()
for i, rollout_idx in enumerate(tqdm(confident_idx[:N_INTERVENTIONS], desc='intervene-v2')):
    info = rollout_info[rollout_idx]
    source_letter = info['pred']
    letters_avail = [l for l in directions_by_letter_v2 if l != source_letter and ord(l)-ord('A') < len(info['options'])]
    if not letters_avail: continue
    target_letter = random.choice(letters_avail)
    
    prompt = format_prompt(info['question'], info['options'])
    forced_prefix = response_tokens_all[rollout_idx][:PATCH_POSITION_V2]
    
    d_source = directions_by_letter_v2[source_letter]
    d_target = directions_by_letter_v2[target_letter]
    patch_vec = PATCH_ALPHA_V2 * (d_target - d_source)
    
    text_baseline = generate_with_patch(prompt, forced_prefix, None, max_new=1024)
    letter_baseline = extract_letter(text_baseline, len(info['options']))
    
    text_patched = generate_with_patch(prompt, forced_prefix, patch_vec, max_new=1024)
    letter_patched = extract_letter(text_patched, len(info['options']))
    
    random_dir = np.random.randn(len(d_source)).astype(np.float32)
    random_dir = random_dir / np.linalg.norm(random_dir) * PATCH_ALPHA_V2
    text_random = generate_with_patch(prompt, forced_prefix, random_dir, max_new=1024)
    letter_random = extract_letter(text_random, len(info['options']))
    
    results_v2.append(dict(
        idx=int(rollout_idx), source=source_letter, target=target_letter,
        baseline=letter_baseline, patched=letter_patched, random=letter_random,
        gold=info['gold'],
    ))
    if (i+1) % 5 == 0:
        n_flip = sum(1 for r in results_v2 if r['patched'] == r['target'])
        n_keep = sum(1 for r in results_v2 if r['patched'] == r['source'])
        print(f'  [V2 {i+1}/{N_INTERVENTIONS}] flipped to target: {n_flip}/{len(results_v2)} | kept source: {n_keep}')

with open(OUT/'intervention_results_v2.json', 'w') as f:
    json.dump(dict(layer=L_PATCH_V2, alpha=PATCH_ALPHA_V2, position=PATCH_POSITION_V2,
                   n=len(results_v2), results=results_v2,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ V2 done in {(time.time()-t0)/60:.1f} min')
